# 🏦 Mini Project 3 — Agentic AI in FinTech
### Comparing Baseline, Single-Agent, and Multi-Agent Architectures


## 🎯 Learning Objectives

1. Understand how tool-calling (function calling) works with LLMs
2. Define your own tool schemas and wire them to real Python functions
3. Build a single-agent system and reason about its limitations
4. Design and implement a multi-agent system — architecture is your choice
5. Build an LLM-as-judge evaluator to score answers automatically
6. Analyse accuracy, hallucination rate, and latency across all architectures and models
7. Reflect critically: *when does added complexity actually pay off?*

---

## 📦 What You Are Given vs What You Build

| Component | Status |
|---|---|
| 5 working tool functions | ✅ Provided |
| JSON schemas for all 7 tools | ✅ Provided |
| `AgentResult` dataclass | ✅ Provided |
| `run_specialist_agent()` loop | ✅ Provided |
| 15 fixed benchmark questions | ✅ Provided |
| Evaluation runner + Excel writer | ✅ Provided |
| **Tool 6: `get_company_overview`** | ❌ You implement |
| **Tool 7: `get_tickers_by_sector`** | ❌ You implement |
| **Baseline** | ❌ You implement |
| **Single-agent system** | ❌ You design + implement |
| **Multi-agent system** | ❌ You design + implement |
| **Evaluator** | ❌ You design + implement |

---

## 🔑 Before You Start

**API keys needed:**
- OpenAI → https://platform.openai.com/api-keys  
- Alpha Vantage (free) → https://www.alphavantage.co/support/#api-key

**Data needed:**
- `sp500_companies.csv` → https://www.kaggle.com/datasets/andrewmvd/sp-500-stocks  
- Unzip and place next to this notebook

**Create a `.env` file** in the same folder (never commit this):
```
OPENAI_API_KEY=sk-proj-...
ALPHAVANTAGE_API_KEY=...
```


## Step 0 — Install & Import

In [ ]:
!pip install openai requests pandas yfinance python-dotenv openpyxl --quiet

In [ ]:
# if running on colab, run this cell:
import os, json, time, sqlite3, requests, textwrap
import pandas as pd
import yfinance as yf
from dataclasses import dataclass, field
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display, Markdown
import os, time, socket, subprocess, requests
from pathlib import Path
from google.colab import userdata

# Clone mock server if needed
if not Path("/content/Mock_AlphaVantage").exists():
    !git clone https://github.com/Phemon/Mock_AlphaVantage.git

# Install dependencies
%cd /content/Mock_AlphaVantage
!pip install -q -r requirements.txt openai python-dotenv yfinance pandas openpyxl

# Load secrets from Colab Secrets
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["ALPHAVANTAGE_API_KEY"] = userdata.get("ALPHAVANTAGE_API_KEY") or "test"

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
ALPHAVANTAGE_API_KEY = os.getenv("ALPHAVANTAGE_API_KEY")

def port_open(host="127.0.0.1", port=2345):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(0.5)
        return s.connect_ex((host, port)) == 0

# Point your notebook tools to the mock server
os.environ["ALPHAVANTAGE_BASE_URL"] = "http://127.0.0.1:2345"
AV_BASE = os.getenv("ALPHAVANTAGE_BASE_URL", "https://www.alphavantage.co")

# Start mock server if needed
if not port_open(port=2345):
    mock_server = subprocess.Popen(
        ["python", "av_mock_server.py"],
        cwd="/content/Mock_AlphaVantage",
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    time.sleep(3)

print("OPENAI_API_KEY loaded:", bool(OPENAI_API_KEY))
print("ALPHAVANTAGE_API_KEY loaded:", bool(ALPHAVANTAGE_API_KEY))
print("AV_BASE =", AV_BASE)
print("Mock server running:", port_open(port=2345))


MODEL_SMALL  = "gpt-4o-mini"
MODEL_LARGE  = "gpt-4o"
ACTIVE_MODEL = MODEL_SMALL          # switch to MODEL_LARGE for the second run
client = OpenAI(api_key=OPENAI_API_KEY)
print(f"✅ Ready  |  active model: {ACTIVE_MODEL}")


In [ ]:
# if running on local, run this cell:
import os, json, time, sqlite3, requests, textwrap, socket, subprocess
import pandas as pd
import yfinance as yf
from dataclasses import dataclass, field
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display, Markdown

load_dotenv(override=True)
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "YOUR_KEY")
ALPHAVANTAGE_API_KEY = os.getenv("ALPHAVANTAGE_API_KEY", "test")

def port_open(host="127.0.0.1", port=2345):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(0.5)
        return s.connect_ex((host, port)) == 0

if not port_open():
    subprocess.Popen(
        ["../.venv/bin/python", "av_mock_server.py"],
        cwd="Mock_AlphaVantage",
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    time.sleep(3)

os.environ["ALPHAVANTAGE_BASE_URL"] = "http://127.0.0.1:2345"
AV_BASE = os.getenv("ALPHAVANTAGE_BASE_URL", "https://www.alphavantage.co")

MODEL_SMALL = "gpt-4o-mini"
MODEL_LARGE = "gpt-4o"
ACTIVE_MODEL = MODEL_SMALL

client = OpenAI(api_key=OPENAI_API_KEY)
print(f"✅ Ready  |  active model: {ACTIVE_MODEL}")
print(f"✅ AV_BASE: {AV_BASE}")


✅ Ready  |  active model: gpt-4o-mini
✅ AV_BASE: http://127.0.0.1:2345


## Step 1 — Build the Local Database

Run `create_local_database()` once after placing `sp500_companies.csv` next to this notebook.  
The function prints all distinct **sector** values — you will need these when implementing Tool 7.


In [116]:
DB_PATH = "./mini-Project-3/stocks.db"

def create_local_database(csv_path: str = "./mini-Project-3/sp500_companies.csv"):
    if not os.path.exists(csv_path):
        raise FileNotFoundError(
            f"'{csv_path}' not found.\n"
            "Download from: https://www.kaggle.com/datasets/andrewmvd/sp-500-stocks"
        )
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip().str.lower()
    df = df.rename(columns={
        "symbol":"ticker", "shortname":"company",
        "sector":"sector",  "industry":"industry",
        "exchange":"exchange", "marketcap":"market_cap_raw"
    })
    def cap_bucket(v):
        try:
            v = float(v)
            return "Large" if v >= 10_000_000_000 else "Mid" if v >= 2_000_000_000 else "Small"
        except: return "Unknown"
    df["market_cap"] = df["market_cap_raw"].apply(cap_bucket)
    df = (df.dropna(subset=["ticker","company"])
            .drop_duplicates(subset=["ticker"])
            [["ticker","company","sector","industry","market_cap","exchange"]])
    conn = sqlite3.connect(DB_PATH)
    df.to_sql("stocks", conn, if_exists="replace", index=False)
    conn.execute("CREATE UNIQUE INDEX IF NOT EXISTS idx_ticker ON stocks(ticker)")
    conn.commit()
    n = pd.read_sql_query("SELECT COUNT(*) AS n FROM stocks", conn).iloc[0]["n"]
    print(f"✅ {n} companies loaded into stocks.db")
    print("\nDistinct sector values stored in DB:")
    print(pd.read_sql_query("SELECT DISTINCT sector FROM stocks ORDER BY sector", conn).to_string(index=False))
    conn.close()

create_local_database()

✅ 502 companies loaded into stocks.db

Distinct sector values stored in DB:
                sector
       Basic Materials
Communication Services
     Consumer Cyclical
    Consumer Defensive
                Energy
    Financial Services
            Healthcare
           Industrials
           Real Estate
            Technology
             Utilities


## Step 2 — Tool Functions

### Provided Tools (5 of 7)

Read each function carefully — you need to understand their return shapes before writing agents.


In [133]:
# ── Tool 1 ── Provided ────────────────────────────────────────
def get_price_performance(tickers: list, period: str = "1y") -> dict:
    """
    % price change for a list of tickers over a period.
    Valid periods: '1mo', '3mo', '6mo', 'ytd', '1y'
    Returns: { TICKER: {start_price, end_price, pct_change, period} }
    """
    results = {}
    for ticker in tickers:
        try:
            data = yf.download(ticker, period=period, progress=False, auto_adjust=True)
            if data.empty:
                results[ticker] = {"error": "No data — possibly delisted"}
                continue
            start = float(data["Close"].iloc[0].item())
            end   = float(data["Close"].iloc[-1].item())
            results[ticker] = {
                "start_price": round(start, 2),
                "end_price"  : round(end,   2),
                "pct_change" : round((end - start) / start * 100, 2),
                "period"     : period,
            }
        except Exception as e:
            results[ticker] = {"error": str(e)}
    return results

# ── Tool 2 ── Provided ────────────────────────────────────────
def get_market_status() -> dict:
    """Open / closed status for global stock exchanges."""
    # return requests.get(
    #     f"https://www.alphavantage.co/query?function=MARKET_STATUS"
    #     f"&apikey={ALPHAVANTAGE_API_KEY}", timeout=10
    # ).json()
    return requests.get(
        f"{AV_BASE}/query?function=MARKET_STATUS"
        f"&apikey={ALPHAVANTAGE_API_KEY}", timeout=10
    ).json()

# ── Tool 3 ── Provided ────────────────────────────────────────
def get_top_gainers_losers() -> dict:
    """Today's top gaining, top losing, and most active tickers."""
    # return requests.get(
    #     f"https://www.alphavantage.co/query?function=TOP_GAINERS_LOSERS"
    #     f"&apikey={ALPHAVANTAGE_API_KEY}", timeout=10
    # ).json()
    return requests.get(
        f"{AV_BASE}/query?function=TOP_GAINERS_LOSERS"
        f"&apikey={ALPHAVANTAGE_API_KEY}", timeout=10
    ).json()

# ── Tool 4 ── Provided ────────────────────────────────────────
def get_news_sentiment(ticker: str, limit: int = 5) -> dict:
    """
    Latest headlines + Bullish / Bearish / Neutral sentiment for a ticker.
    Returns: { ticker, articles: [{title, source, sentiment, score}] }
    """
    # data = requests.get(
    #     f"https://www.alphavantage.co/query?function=NEWS_SENTIMENT"
    #     f"&tickers={ticker}&limit={limit}&apikey={ALPHAVANTAGE_API_KEY}", timeout=10
    # ).json()
    data = requests.get(
        f"{AV_BASE}/query?function=NEWS_SENTIMENT"
        f"&tickers={ticker}&limit={limit}&apikey={ALPHAVANTAGE_API_KEY}", timeout=10
    ).json()
    return {
        "ticker": ticker,
        "articles": [
            {
                "title"    : a.get("title"),
                "source"   : a.get("source"),
                "sentiment": a.get("overall_sentiment_label"),
                "score"    : a.get("overall_sentiment_score"),
            }
            for a in data.get("feed", [])[:limit]
        ],
    }

# ── Tool 5 ── Provided ────────────────────────────────────────
def query_local_db(sql: str) -> dict:
    """
    Run any SQL SELECT on stocks.db.
    Table 'stocks' columns: ticker, company, sector, industry, market_cap, exchange
    market_cap values: 'Large' | 'Mid' | 'Small'
    """
    try:
        conn = sqlite3.connect(DB_PATH)
        df   = pd.read_sql_query(sql, conn)
        conn.close()
        return {"columns": list(df.columns), "rows": df.to_dict(orient="records")}
    except Exception as e:
        return {"error": str(e)}

print("✅ 5 provided tools ready")

✅ 5 provided tools ready


---
## 🛠️ Task 1 — Implement the 2 Missing Tools (20 pts)

### Tool 6 — `get_company_overview`

Call the Alpha Vantage **OVERVIEW** endpoint for a single ticker.  
Docs: https://www.alphavantage.co/documentation/#company-overview  

Required return format:
```python
{
    "ticker"    : str,   # the input ticker
    "name"      : str,   # company full name
    "sector"    : str,
    "pe_ratio"  : str,   # field name in API: PERatio
    "eps"       : str,   # field name: EPS
    "market_cap": str,   # field name: MarketCapitalization
    "52w_high"  : str,   # field name: 52WeekHigh
    "52w_low"   : str,   # field name: 52WeekLow
}
```
If the API returns no `"Name"` key (rate-limited or invalid ticker), return:
```python
{"error": f"No overview data for {ticker}"}
```

---

### Tool 7 — `get_tickers_by_sector`

Query `stocks.db` for companies matching a sector **or** industry.

**Critical detail:** Look at the sector values printed by `create_local_database()`.  
The DB stores broad sectors in `sector` (e.g. `"Information Technology"`) and  
specific industries in `industry` (e.g. `"Semiconductors"`).  
A query for `"semiconductor"` must fall back to the `industry` column — otherwise it returns zero rows.

Required logic:
1. Try exact match on `sector` column (case-insensitive)  
2. If 0 results → try `LIKE '%sector%'` on the `industry` column  

Required return format:
```python
{
    "sector": str,          # the input search term
    "stocks": [
        {"ticker": str, "company": str, "industry": str},
        ...
    ]
}
```


In [122]:
# ── Tool 6 — YOUR IMPLEMENTATION ─────────────────────────────
def get_company_overview(ticker: str) -> dict:
    ### YOUR CODE HERE
    keys = ["ticker", "name", "sector", "pe_ratio", "eps", "market_cap", "52w_high", "52w_low"]
    val_names = ["Symbol", "Name", "Sector", "PERatio","EPS","MarketCapitalization", "52WeekHigh","52WeekLow"]
    try: 
        # data = requests.get(f'https://www.alphavantage.co/query?function=OVERVIEW&symbol={ticker}&apikey={ALPHAVANTAGE_API_KEY}', timeout=10).json()
        data = requests.get(f'{AV_BASE}/query?function=OVERVIEW&symbol={ticker}&apikey={ALPHAVANTAGE_API_KEY}', timeout=10).json()
        if not data.get("Name"):
            return {"error": f"No overview data for {ticker}"}
        return {key: data.get(val) for key, val in zip(keys, val_names)}
    except Exception as e:
        return {"error": str(e)}


    

# ── Tool 7 — YOUR IMPLEMENTATION ─────────────────────────────
def get_tickers_by_sector(sector: str) -> dict:
    ### YOUR CODE HERE
    sql_sector = """
        SELECT ticker, company, sector, industry
        FROM stocks
        WHERE LOWER(sector) LIKE Lower(?)
    """ 
    sql_industry = """
        SELECT ticker, company, sector, industry
        FROM stocks
        WHERE LOWER(industry) LIKE Lower(?)
    """
    try:
        with sqlite3.connect(DB_PATH) as conn:
            df   = pd.read_sql_query(sql_sector, conn, params=[f"%{sector}%"])
            if df.empty:
                df = pd.read_sql_query(sql_industry, conn, params=[f"%{sector}%"])
        return {
            "sector": df["sector"].iloc[0], 
            "stocks":df[["ticker", "company", "industry"]].to_dict(orient="records")
            }
    except Exception as e:
        return {"error": str(e)}

# ── Automated tests — all assertions must pass ────────────────
print("=== Tool 6 tests ===")
aapl = get_company_overview("AAPL")
assert "pe_ratio" in aapl,             "Missing pe_ratio key"
assert aapl.get("ticker") == "AAPL",   "ticker key wrong"
assert aapl.get("name"),               "name should not be empty"
print(f"  AAPL P/E: {aapl['pe_ratio']} ✅")

bad = get_company_overview("INVALIDTICKER999")
assert "error" in bad,  "Invalid ticker should return error key"
print(f"  Invalid ticker handled correctly ✅")

print("\n=== Tool 7 tests ===")
tech = get_tickers_by_sector("Information Technology")
assert len(tech["stocks"]) > 0,        "Should find IT stocks (exact sector match)"
print(f"  'Information Technology' → {len(tech['stocks'])} stocks ✅")

semi = get_tickers_by_sector("semiconductor")
assert len(semi["stocks"]) > 0,        "Should find via industry fallback (LIKE match)"
print(f"  'semiconductor' (industry fallback) → {len(semi['stocks'])} stocks ✅")

print("\n✅ All tool tests passed")

=== Tool 6 tests ===
  AAPL P/E: 32.374683 ✅
  Invalid ticker handled correctly ✅

=== Tool 7 tests ===
  'Information Technology' → 11 stocks ✅
  'semiconductor' (industry fallback) → 18 stocks ✅

✅ All tool tests passed


## Step 3 — Tool Schemas (Provided)

Schemas tell the LLM what tools exist, what they do, and what arguments they take.  
You do not modify these — but you will reference the schema lists when building your agents.

**Key concept:** You can give different agents different *subsets* of schemas.  
A specialist that only sees 2–3 relevant schemas makes fewer wrong tool choices  
than one that sees all 7.


In [125]:
def _s(name, desc, props, req):
    return {"type":"function",
            "function":{
                "name":name,"description":desc,
                "parameters":{"type":"object","properties":props,"required":req}
                }
            }

SCHEMA_TICKERS  = _s("get_tickers_by_sector",
    "Return all stocks in a sector or industry from the local database. "
    "Use broad sector names ('Information Technology', 'Energy') or sub-sectors ('semiconductor', 'insurance').",
    {"sector":{"type":"string","description":"Sector or industry name"}}, ["sector"])

SCHEMA_PRICE    = _s("get_price_performance",
    "Get % price change for a list of tickers over a time period. "
    "Periods: '1mo','3mo','6mo','ytd','1y'.",
    {"tickers":{"type":"array","items":{"type":"string"}},
     "period":{"type":"string","default":"1y"}}, ["tickers"])

SCHEMA_OVERVIEW = _s("get_company_overview",
    "Get fundamentals for one stock: P/E ratio, EPS, market cap, 52-week high and low.",
    {"ticker":{"type":"string","description":"Ticker symbol e.g. 'AAPL'"}}, ["ticker"])

SCHEMA_STATUS   = _s("get_market_status",
    "Check whether global stock exchanges are currently open or closed.", {}, [])

SCHEMA_MOVERS   = _s("get_top_gainers_losers",
    "Get today's top gaining, top losing, and most actively traded stocks.", {}, [])

SCHEMA_NEWS     = _s("get_news_sentiment",
    "Get latest news headlines and Bullish/Bearish/Neutral sentiment scores for a stock.",
    {"ticker":{"type":"string"},"limit":{"type":"integer","default":5}}, ["ticker"])

SCHEMA_SQL      = _s("query_local_db",
    "Run a SQL SELECT on stocks.db. "
    "Table 'stocks': ticker, company, sector, industry, market_cap (Large/Mid/Small), exchange.",
    {"sql":{"type":"string","description":"A valid SQL SELECT statement"}}, ["sql"])

# All 7 schemas in one list — used by single agent
ALL_SCHEMAS = [SCHEMA_TICKERS, SCHEMA_PRICE, SCHEMA_OVERVIEW,
               SCHEMA_STATUS, SCHEMA_MOVERS, SCHEMA_NEWS, SCHEMA_SQL]

# Dispatch map — maps the tool name string the LLM returns → the Python function to call
ALL_TOOL_FUNCTIONS = {
    "get_tickers_by_sector" : get_tickers_by_sector,
    "get_price_performance"  : get_price_performance,
    "get_company_overview"   : get_company_overview,
    "get_market_status"      : get_market_status,
    "get_top_gainers_losers" : get_top_gainers_losers,
    "get_news_sentiment"     : get_news_sentiment,
    "query_local_db"         : query_local_db,
}

print("✅ Schemas ready")
print(f"   Tools available: {list(ALL_TOOL_FUNCTIONS.keys())}")

✅ Schemas ready
   Tools available: ['get_tickers_by_sector', 'get_price_performance', 'get_company_overview', 'get_market_status', 'get_top_gainers_losers', 'get_news_sentiment', 'query_local_db']


## Step 4 — AgentResult and Base Runner (Provided)

`AgentResult` is the standard return type for every agent — baseline, single, and all multi-agent specialists.  
`run_specialist_agent` is the reusable tool-call loop that every agent uses.  
Study this loop carefully before writing your own agents — it is what connects the LLM's tool requests to your Python functions.


In [126]:
@dataclass
class AgentResult:
    agent_name   : str
    answer       : str
    tools_called : list  = field(default_factory=list)   # tool names in order called
    raw_data     : dict  = field(default_factory=dict)   # tool name → raw tool output
    confidence   : float = 0.0                           # set by evaluator / critic
    issues_found : list  = field(default_factory=list)   # set by evaluator / critic
    reasoning    : str   = ""

    def summary(self):

        summary = f"{'─'*54}\n\n"
        summary += f"**Agent**      : {self.agent_name}\n\n"
        summary += f"**Tools used** : {', '.join(self.tools_called) or 'none'}\n\n"
        summary += f"**Confidence** : {self.confidence:.0%}\n\n"
        # print(f"\n{'─'*54}")
        # print(f"Agent      : {self.agent_name}")
        # print(f"Tools used : {', '.join(self.tools_called) or 'none'}")
        # print(f"Confidence : {self.confidence:.0%}")
        if self.issues_found:
            summary += f"**Issues**     : {'; '.join(self.issues_found)}\n\n"
            # print(f"Issues     : {'; '.join(self.issues_found)}")
        summary += f"{'─'*54}\n\n"
        summary += f"**Answer**     :\n\n{self.answer}"
        # print(f"Answer     :\n{textwrap.indent(self.answer[:500], '  ')}")
        # display(Markdown(summary))
        return summary

print("✅ AgentResult defined")

✅ AgentResult defined


In [127]:
def run_specialist_agent(
    agent_name   : str,
    system_prompt: str,
    task         : str,
    tool_schemas : list,
    max_iters    : int  = 8,
    verbose      : bool = True,
) -> AgentResult:
    """
    Core agentic loop used by every agent in this project.

    How it works:
      1. Sends system_prompt + task to the LLM
      2. If the LLM requests a tool call → looks up the function in ALL_TOOL_FUNCTIONS,
         executes it, appends the result to the message history, loops back to step 1
      3. When the LLM produces a response with no tool calls → returns an AgentResult

    Parameters
    ----------
    agent_name    : display name for logging
    system_prompt : the agent's persona, rules, and focus area
    task          : the specific question or sub-task for this agent
    tool_schemas  : list of schema dicts this agent is allowed to use
                    (pass [] for no tools — used by baseline)
    max_iters     : hard cap on iterations to prevent infinite loops
    verbose       : print each tool call as it happens
    """
    ### YOUR CODE HERE ###
    tools_called = []
    raw_data = {}
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": task}
    ]

    for i in range(max_iters):
        params = {
            "model":ACTIVE_MODEL,
            "messages": messages,
            "temperature": 0,
        }

        if tool_schemas:
            params["tools"] = tool_schemas
            params["tool_choice"] = "auto"
        
        response = client.chat.completions.create(**params)
        output = response.choices[0].message
        has_tool_call = output.tool_calls or []

        if not has_tool_call:
            answer = (output.content or "").strip()
            return AgentResult(
                agent_name=agent_name,
                answer=answer or "No answer generated",
                tools_called=tools_called,
                raw_data=raw_data
            )
        
        messages.append(output.model_dump(exclude_none=True))
        for tc in has_tool_call:
            name = tc.function.name
            arguments = tc.function.arguments or "{}"
            try:
                args = json.loads(arguments)
            except Exception:
                args = {}
            
            if verbose:
                print(f"{agent_name} is calling tool: {name}")

            func = ALL_TOOL_FUNCTIONS.get(name)
            if func:
                f_output = func(**args)
            else:
                f_output = {"Error": f"Function: {name} is not avaliable"}
            
            tools_called.append(name)
            raw_data.setdefault(name, []).append(f_output)
            results = json.dumps(f_output)

            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": str(results),
                }
            )

    return AgentResult(
        agent_name=agent_name,
        answer=f"Maximum iteration {max_iters} reached",
        tools_called=tools_called,
        raw_data=raw_data,
        issues_found=["maximum iteration reached"]
    )


print("✅ run_specialist_agent ready")

✅ run_specialist_agent ready


---
## 🛠️ Task 2 — Implement the Baseline (10 pts)

The baseline is a **single LLM call with no tools**. The model answers entirely from its training knowledge.

This is your **control condition**. Every architecture you build should be compared against it.  
If agents don't outperform the baseline, there's no justification for the extra complexity.

**Requirements:**
- One call to `client.chat.completions.create()` — no `tools` argument
- Return `AgentResult(agent_name="Baseline", answer=..., tools_called=[])`
- Use a sensible system prompt that encourages the model to be honest about uncertainty

**Grading checks:**
- `tools_called` must be `[]`
- Answer must not be empty


In [108]:
def run_baseline(question: str, verbose: bool = True) -> AgentResult:
    # Implement a single LLM call with no tools.
    # Use run_specialist_agent() with an empty tool_schemas list — or make the call directly.
    # Return an AgentResult with agent_name="Baseline" and tools_called=[].
    ### YOUR CODE HERE
    system_prompt = (
        "You are a finance assistant. "
        "Answer clearly and concisely. No follow-ups."
        "If you are unsure or data may be outdated, say so explicitly."
    )
    return run_specialist_agent(
        agent_name="Baseline Agent",
        system_prompt=system_prompt,
        task=question,
        tool_schemas=[],
        max_iters=1,
        verbose=verbose
    )

# Quick test
bl = run_baseline("What is Apple's approximate P/E ratio?", verbose=True)
assert bl.tools_called == [], "Baseline must not call any tools"
assert bl.answer, "Answer must not be empty"
bl.summary()

"──────────────────────────────────────────────────────\n\n**Agent**      : Baseline Agent\n\n**Tools used** : none\n\n**Confidence** : 0%\n\n──────────────────────────────────────────────────────\n\n**Answer**     :\n\nAs of October 2023, Apple's approximate P/E ratio is around 28. However, please verify with a current financial source, as this figure can fluctuate frequently."

---
## 🛠️ Task 3 — Design and Implement the Single Agent (20 pts)

A single agent is **one LLM with access to all 7 tools**. Everything — planning which tools to call, executing them, and synthesising the answer — happens in one context window.

You decide how to build it. The guidance below is a starting point, not a recipe.

---

### Things to think about when writing your system prompt

**Role and scope** — what is this agent's job? Being specific helps the model stay focused.

**Accuracy rules** — how should the agent behave when a tool returns an error or empty data?  
This is critical: an agent with no rules tends to fabricate plausible-looking numbers when the API fails.

**Multi-step reasoning** — some questions require chaining tools. For example:  
*"Which energy stocks grew the most this year?"* requires first looking up energy tickers,  
then fetching price data for each one. Without explicit guidance, single agents often  
skip the lookup step and guess tickers instead.

**Tool selection** — the agent sees all 7 tools. Giving it rules about *when* to use each  
(e.g. "use `query_local_db` with SQL when you need to filter by sector or market cap")  
reduces wrong tool choices.

---

### Recommended structure

```python
SINGLE_AGENT_PROMPT = """
<your system prompt here>
"""

def run_single_agent(question: str, verbose: bool = True) -> AgentResult:
    # Call run_specialist_agent() with:
    #   agent_name    = "Single Agent"
    #   system_prompt = SINGLE_AGENT_PROMPT
    #   task          = question
    #   tool_schemas  = ALL_SCHEMAS   (all 7)
    #   max_iters     = 10
    # Return the AgentResult directly.
    pass
```

---

### Test before you move on

Run the three cells below and check the results make sense before building the multi-agent system.


In [135]:
# ── YOUR SINGLE AGENT IMPLEMENTATION ─────────────────────────
# Write your system prompt and run_single_agent() function here.
# Comments above explain what to think about — the implementation is yours.

### YOUR CODE HERE
SINGLE_AGENT_PROMPT = """
You are a finance analysis agent with access to 7 tools.
Your job is to answer accurately using tool evidence, not guesses.

Rules:
1) Never invent numbers, tickers, prices, P/E, returns, or sentiment.
2) If data is missing or a tool returns an error, state that clearly.
3) For any claim involving current market data or specific numeric values, call tools first.
4) Always create an explicit plan before finalizing: reason through how to break down the question, decide which tools to call, and specify the call order. State this plan clearly.
5) Keep answers concise, structured, and grounded in returned tool outputs.
6) If data remains insufficient after reasonable tool use, say what is missing and why.

Sufficiency gate (run this check every turn):
1) Check whether existing tool results are sufficient to answer the exact question with chain-of-thought reasoning.
2) Confirm all required conditions are satisfied (time period, filters, ranking, constraints etc.).
3) If sufficient: do not call more tools; provide final answer without reasoning from previous steps.
4) If not sufficient: identify the missing info and call the most appropriate next tool.
5) After each new tool result, re-run this sufficiency gate.

Tool usage policy:
- get_tickers_by_sector:
  Use when the user asks about a sector/industry (e.g., energy, semiconductor, finance, tech stocks in DB). Returns all stocks in a sector or industry from the local database.
- query_local_db:
  Use for custom filtering/counting in stocks.db. Fields include ticker, company, sector, industry, market_cap (Large/Mid/Small), exchange. Make sure to use A valid SQL SELECT statement as input.
- get_price_performance:
  Use for return comparisons over periods. Returns % price change for tickers over a period ('1mo','3mo','6mo','ytd','1y').
- get_company_overview:
  Use for company fundamentals (P/E, EPS, market cap, 52-week high/low).
- get_news_sentiment:
  Use for sentiment/headlines for specific tickers.
- get_market_status:
  Use when asked whether markets/exchanges are open.
- get_top_gainers_losers:
  Use for today’s top gainers/losers/most active.

Reasoning workflow:
A) Identify exactly what the question asks.
B) Choose the minimum required tools.
C) If needed, chain tools (example: sector -> tickers -> price performance -> rank).
D) Step by step, verify that computed results satisfy all question constraints before final answer.
E) Provide final answer with key results and brief caveats.

Output style:
- For the final answer output, give a concise direct answer, no reasoning needed from previous steps.
- Then provide a short evidence section (tools used and key values).
- If uncertain, explicitly state what is unknown and why.
"""



In [16]:
def run_single_agent(question: str, verbose: bool = True) -> AgentResult:
    return run_specialist_agent(
        agent_name="Single Agent",
        system_prompt=SINGLE_AGENT_PROMPT,
        task=question,
        tool_schemas=ALL_SCHEMAS,
        max_iters=10,
        verbose=verbose,
    )

In [81]:
# Test 1 — easy question, 1 tool expected
r1 = run_single_agent("What is the P/E ratio of Apple (AAPL)?")
r1.summary()

Single Agent is calling tool: get_company_overview


──────────────────────────────────────────────────────

**Agent**      : Single Agent

**Tools used** : get_company_overview

**Confidence** : 0%

──────────────────────────────────────────────────────

**Answer**     :

The P/E ratio of Apple Inc. (AAPL) is 33.23.

**Evidence:**
- Tool used: `get_company_overview`
- Key value: P/E ratio = 33.23

In [105]:
# Test 2 — medium question, requires sector lookup + price fetch
r2 = run_single_agent("Which energy stocks in the database had the best 6-month performance?")
r2.summary()

Single Agent is calling tool: get_tickers_by_sector
Single Agent is calling tool: get_price_performance


$HES: possibly delisted; no price data found  (period=6mo) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['HES']: possibly delisted; no price data found  (period=6mo) (Yahoo error = "No data found, symbol may be delisted")


──────────────────────────────────────────────────────

**Agent**      : Single Agent

**Tools used** : get_tickers_by_sector, get_price_performance

**Confidence** : 0%

──────────────────────────────────────────────────────

**Answer**     :

The energy stocks with the best 6-month performance are as follows:

1. **Texas Pacific Land Corporation (TPL)**: 72.55%
2. **Targa Resources, Inc. (TRGP)**: 49.89%
3. **Valero Energy Corporation (VLO)**: 46.72%
4. **APA Corporation (APA)**: 48.49%
5. **Halliburton Company (HAL)**: 58.68%

### Evidence
- **Top Performers**:
  - TPL: 72.55%
  - TRGP: 49.89%
  - VLO: 46.72%
  - APA: 48.49%
  - HAL: 58.68%

- **Tools Used**: 
  - `get_tickers_by_sector` for retrieving energy stocks.
  - `get_price_performance` for 6-month performance data.

In [107]:
# Test 3 — hard multi-condition question
r3 = run_single_agent("Top 3 tech stocks that dropped this month but grew this year.")
r3.summary()

Single Agent is calling tool: get_tickers_by_sector
Single Agent is calling tool: get_price_performance


$FI: possibly delisted; no price data found  (period=1mo) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['FI']: possibly delisted; no price data found  (period=1mo) (Yahoo error = "No data found, symbol may be delisted")


Single Agent is calling tool: get_price_performance


$FI: possibly delisted; no price data found  (period=ytd) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['FI']: possibly delisted; no price data found  (period=ytd) (Yahoo error = "No data found, symbol may be delisted")


──────────────────────────────────────────────────────

**Agent**      : Single Agent

**Tools used** : get_tickers_by_sector, get_price_performance, get_price_performance

**Confidence** : 0%

──────────────────────────────────────────────────────

**Answer**     :

The analysis of tech stocks that dropped this month but grew this year is as follows:

### Stocks Performance Summary:
1. **ACN (Accenture plc)**
   - 1 Month Change: -11.06%
   - YTD Change: -17.68%

2. **IBM (International Business Machines)**
   - 1 Month Change: -13.69%
   - YTD Change: -11.49%

3. **FIS (Fidelity National Information Services)**
   - 1 Month Change: -2.6%
   - YTD Change: -22.92%

4. **CTSH (Cognizant Technology Solutions)**
   - 1 Month Change: -14.23%
   - YTD Change: -18.66%

5. **CDW (CDW Corporation)**
   - 1 Month Change: -12.4%
   - YTD Change: -6.18%

6. **LDOS (Leidos Holdings, Inc.)**
   - 1 Month Change: -8.98%
   - YTD Change: -4.26%

7. **EPAM (EPAM Systems, Inc.)**
   - 1 Month Change: -22.55%
   - YTD Change: -27.88%

8. **JKHY (Jack Henry & Associates, Inc.)**
   - 1 Month Change: -3.29%
   - YTD Change: -6.0%

### Conclusion:
None of the tech stocks analyzed have shown a positive YTD performance while also experiencing a drop this month. All listed stocks have negative YTD changes.

### Evidence:
- Tools Used: `get_tickers_by_sector`, `get_price_performance`
- Key Values: All stocks analyzed had negative YTD performance. 

### Missing Information:
No stocks met the criteria of dropping this month while having a positive YTD growth.

In [17]:
# Test 4 — easy-condition question
r4 = run_single_agent("What is NVIDIA's stock price performance over the last month?")
r4.summary()

Single Agent is calling tool: get_price_performance


──────────────────────────────────────────────────────

**Agent**      : Single Agent

**Tools used** : get_price_performance

**Confidence** : 0%

──────────────────────────────────────────────────────

**Answer**     :

NVIDIA's stock price performance over the last month shows a percentage change of -6.43%. The stock started at $190.04 and ended at $177.82.

**Evidence:**
- Ticker: NVDA
- Start Price: $190.04
- End Price: $177.82
- Percentage Change: -6.43%

In [19]:
# Test 5 — medium question
r5 = run_single_agent("What is the news sentiment for Tesla (TSLA) and how has its stock moved this month?")
r5.summary()

Single Agent is calling tool: get_news_sentiment
Single Agent is calling tool: get_price_performance


──────────────────────────────────────────────────────

**Agent**      : Single Agent

**Tools used** : get_news_sentiment, get_price_performance

**Confidence** : 0%

──────────────────────────────────────────────────────

**Answer**     :

**Final Answer:**

The news sentiment for Tesla (TSLA) includes a mix of articles with varying sentiments, with the most notable being:
- Somewhat-Bearish sentiment regarding a lawsuit against Tesla.
- Other articles have neutral or somewhat-bullish sentiments.

For the stock performance this month, Tesla (TSLA) has experienced a price decrease of approximately 4.93%, moving from a starting price of $417.32 to an ending price of $396.73.

**Evidence:**
- News Sentiment: Mixed sentiments with some articles being somewhat-bearish and others neutral or somewhat-bullish.
- Price Performance: 1-month change of -4.93% (from $417.32 to $396.73).

---
## 🛠️ Task 4 — Design and Implement a Multi-Agent System (25 pts)

You must build a multi-agent system that answers the same 15 questions.  
**The architecture is your choice.** Experiment, compare, and justify your decision in the reflections.

---

### Three architectures to consider

**Orchestrator + Specialists + Critic**
```
User Question
     │
 Orchestrator  ← reads question, writes a plan, delegates sub-tasks
     │
 ┌───┼───┐
 Ag1 Ag2 Ag3   ← each handles one domain, sees a subset of tools
 └───┼───┘
  Critic        ← fact-checks each agent's answer vs its raw tool data
     │
 Synthesizer    ← merges verified answers into one final response
```
*Good for:* complex cross-domain questions  
*Tradeoff:* most latency, most API calls, but most transparency

---

**Sequential Pipeline**
```
User Question → Agent1 → Agent2 → Agent3 → Final Answer
```
Each agent receives the previous agent's output as context.  
*Good for:* questions with a natural order (find tickers → get prices → summarise)  
*Tradeoff:* errors propagate — a wrong ticker in step 1 breaks all later steps

---

**Parallel Specialists + Aggregator**
```
User Question
  ├── Price Agent       ─┐
  ├── Fundamentals Agent  ├─→ Aggregator → Final Answer
  └── Sentiment Agent   ─┘
```
All specialists run, aggregator merges results.  
*Good for:* speed (can use `ThreadPoolExecutor` to run in parallel)  
*Tradeoff:* no cross-checking between agents

---

### Suggested tool subsets per specialist
These are starting points — adjust based on your design:

```python
MARKET_TOOLS      = [SCHEMA_TICKERS, SCHEMA_PRICE, SCHEMA_STATUS, SCHEMA_MOVERS]
FUNDAMENTAL_TOOLS = [SCHEMA_OVERVIEW, SCHEMA_SQL, SCHEMA_TICKERS]
SENTIMENT_TOOLS   = [SCHEMA_NEWS, SCHEMA_SQL]
```

Giving each specialist only its relevant schemas reduces wrong tool choices.

---

### Required return contract

Whatever architecture you choose, `run_multi_agent()` must return this dict  
(the evaluation runner reads these fields):

```python
{
    "final_answer"  : str,         # the answer shown to the user
    "agent_results" : list,        # list[AgentResult] — one per specialist that ran
    "elapsed_sec"   : float,       # total wall clock time
    "architecture"  : str,         # short name e.g. "orchestrator-critic", "pipeline", "parallel"
}
```

The evaluation runner extracts from `agent_results`:
- `tools_called` (all tools used across specialists)
- `confidence` (averaged across specialists)
- `issues_found` (all issues concatenated)

---

### Document your design choices in document

The grading for this task includes your reasoning — explain in document:
- Which architecture you chose and why
- What you tried that didn't work
- How you divided tools between specialists
- What your verification/confidence mechanism does


In [157]:
# ── YOUR MULTI-AGENT IMPLEMENTATION ──────────────────────────
#
# Architecture chosen: [orchestrator-critic]
# Reason: [explain why you chose this over the alternatives]
#
# Specialist breakdown:
#   Agent 1 — [name, domain, tool subset]
#   Agent 2 — [name, domain, tool subset]
#   Agent N — [name, domain, tool subset]
#
# Verification mechanism: [how does your system check answer quality?]
#
### YOUR CODE HERE
class Orchestrator:
    def __init__(self):
        self.prompt = """
            You are the Orchestrator in a multi-agent stock analysis system.

            Your job is to:
            1. Write a detailed plan for answering the user's question.
            2. Decide which specialists are needed.
            3. Give each selected specialist one concrete instruction based on what that specialist can actually do.

            Available specialists and their real capabilities:

            - market_specialist
            Can do:
            - find tickers by sector or industry
            - get stock price performance over 1mo, 3mo, 6mo, ytd, or 1y
            - compare returns across tickers
            - identify top gainers, losers, and most active stocks
            - check whether markets are open or closed
            Cannot do:
            - retrieve P/E ratio, EPS, market cap, or 52-week high/low
            - retrieve news sentiment

            - fundamental_specialist
            Can do:
            - retrieve company overview data such as P/E ratio, EPS, market cap, 52-week high, and 52-week low
            - query the local stock database for filtering by sector, industry, market_cap, or exchange
            - identify companies/tickers from the local database
            Cannot do:
            - retrieve current or live stock price
            - calculate return or momentum from market price history
            - retrieve news sentiment

            - news_specialist
            Can do:
            - retrieve recent news headlines and sentiment for a ticker
            - use the local database to identify relevant tickers before checking sentiment
            Cannot do:
            - retrieve price performance or current stock price
            - retrieve P/E ratio, EPS, market cap, or 52-week high/low

            Routing rules:
            - Questions about P/E, EPS, market cap, 52-week high, or 52-week low -> fundamental_specialist
            - Questions about price performance, return ranking, gainers/losers, or market status -> market_specialist
            - Questions about headlines, catalysts, or sentiment -> news_specialist
            - If a question spans multiple data types, call multiple specialists
            - If a tool already provides the requested metric directly, ask for that metric directly rather than asking the specialist to derive it manually

            Important examples:
            - If the user asks for P/E ratio, send it to fundamental_specialist and ask it to retrieve the returned pe_ratio directly
            - Do not ask fundamental_specialist for current market price
            - If the user asks for current or recent stock movement, send it to market_specialist
            - If the user asks for both return and P/E ratio, use both market_specialist and fundamental_specialist
            - If the user asks for sentiment plus returns, use both news_specialist and market_specialist

            Rules:
            - Choose the minimum number of specialists needed
            - Do not duplicate work across specialists
            - Each instruction must be specific, concise, and executable with that specialist's tools
            - Do not answer the user directly
            - If the user is only greeting, making small talk, or asking something that does not require stock-analysis tools, return an empty specialists_to_call list
            - Do not ask follow-up questions
        """


    def run(self, question: str, conv_hist: str=""):
        cxt =(
             f"question:\n{question}\n\n"
             f"conversation history:\n{conv_hist}"
        )
        messages = [
            {"role":"system", "content": self.prompt},
            {"role": "user", "content": cxt},
        ]
        response_schema = {
            "type": "json_schema",
            "json_schema":{
                "name": "orchestrator_result",
                "strict": True,
                "schema": {
                    "type": "object",
                    "properties":{
                        "plan": {"type": "string"},
                        "specialists_to_call":{
                            "type":"array",
                            "items": {
                                "type":"object",
                                "properties":{
                                    "agent_name": {"type": "string", "enum":["market_specialist", "fundamental_specialist", "news_specialist"]},
                                    "instruction": {"type": "string"},
                                },
                            "required": ["agent_name", "instruction"],
                            "additionalProperties": False,
                            }
                        },
                    },
                "required":["plan", "specialists_to_call"],
                "additionalProperties": False,
                },
            },
        }

        params = {
            "model":ACTIVE_MODEL,
            "messages": messages,
            "response_format": response_schema,
            "temperature": 0,
        }
        response = client.chat.completions.create(**params)
        output = response.choices[0].message.content
        results = json.loads((output or "").strip())

        return AgentResult(
            agent_name="Orchestrator",
            answer=results["plan"],
            raw_data={"specialist_to_call": results["specialists_to_call"]},
        )
        

class Specialist:
    def __init__(self, name, schema):
        self.name = name
        self.prompt = """ """
        self.schema = schema

    def run(self, task: str, cxt: str=""):
        task += cxt
        specialist_results = run_specialist_agent(
            agent_name=self.name,
            system_prompt=self.prompt,
            task=task,
            tool_schemas=self.schema,
            max_iters=5,
            verbose=True,
        )
        
        return specialist_results

class Critic:
    def __init__(self):
        self.prompt = """
            You are a critic reviewing another specialist agent's output.
            Decide whether the specialist answer is acceptable based only on the provided task, answer, raw_data and issues it found.
            Return JSON with:
            - judgement: 1 if the answer is acceptable, 0 otherwise
            - reasoning: brief explanation why the specialist agent answer is pass or fail
            - confidence: Set confidence as a number from 0 to 1 representing how certain you are that your judgment is correct.
                Use higher confidence when the evidence in raw_data clearly supports your judgment.
                Use lower confidence when the evidence is incomplete, ambiguous, contradictory, or hard to verify.
                Do not use confidence to express whether the answer is good or bad; use it only to express certainty in your evaluation.

            Be strict. Do not invent evidence not present in the input. Do not follow-up.
        """
    def run(self, task, specalist_results: AgentResult):
        specialist_response = {
            "task":task,
            "specialist": specalist_results.agent_name,
            "answer": specalist_results.answer,
            "raw_data": specalist_results.raw_data,
            "issues_found":specalist_results.issues_found,
        }
        messages = [
            {"role":"system", "content": self.prompt},
            {"role": "user", "content": json.dumps(specialist_response)},
        ]
        response_schema = {
            "type": "json_schema",
            "json_schema": {
                "name": "critic_result",
                "strict": True,
                "schema": {
                    "type": "object",
                    "properties": {
                        "judgement": {"type": "number"},
                        "reasoning": {"type": "string"},
                        "confidence": {"type": "number"},
                    },
                    "required": ["judgement", "reasoning", "confidence"],
                    "additionalProperties": False,
                },
            },
        }

        params = {
            "model": ACTIVE_MODEL,
            "messages": messages,
            "response_format": response_schema,
            "temperature": 0
        }
        response = client.chat.completions.create(**params)
        output = response.choices[0].message.content
        results = json.loads((output or "").strip())

        return AgentResult(
            agent_name="Critic",
            answer=(int(results["judgement"])==1),
            confidence=results["confidence"],
            reasoning=results["reasoning"],
        )

class Synthsizer:
    def __init__(self):
        self.prompt = """
            You are the Synthesizer in a multi-agent stock analysis system.

            You will receive a JSON input containing:
            - the user's original question
            - the orchestrator's plan
            - a list of validated specialist results

            The specialist results have already passed critic review, but they still may be incomplete, partially relevant, or mutually contradictory.

            Your job is to decide whether the validated results are sufficient to answer the user's question.

            Output rules:
            - Always return structured JSON only.
            - Return exactly these fields:
            - confidence
            - answer
            - reasoning

            Interpret confidence as a binary sufficiency flag:
            - confidence = 1 means the validated results are sufficient to answer the user's question
            - confidence = 0 means the validated results are not sufficient to answer the user's question

            Meaning of answer:
            - If confidence = 1, answer must be the final user-facing answer
            - If confidence = 0, answer must be a replanning instruction or suggestion for the orchestrator describing what additional information or specialist work is needed

            Meaning of reasoning:
            - Explain why the validated results are sufficient or insufficient
            - Include any important missing information, unresolved conflicts, or gaps in evidence

            Decision rules:
            - Set confidence = 1 only if the validated results are enough to answer the user's question directly and responsibly
            - Set confidence = 0 if key required evidence is missing, if the results do not actually answer the question, or if contradictions prevent a reliable answer
            - Contradictions should only cause failure if they materially affect the conclusion
            - If the available evidence is only partial and not enough for a reliable final answer, set confidence = 0
            - Do not invent facts, numbers, or claims not present in the input
            - Use only the validated specialist results as evidence
            - Do not mention internal roles such as orchestrator, specialist, critic, or synthesizer in the final user-facing answer

            If confidence = 1:
            - combine the validated results into one clear, concise, coherent final answer

            If confidence = 0:
            - do not try to fully answer the user's question
            - use answer to tell the orchestrator exactly how the plan should be refined
            - be specific about what additional evidence, tool usage, or specialist routing is needed
        """


    def run(self, question:str, plan, valid_results: dict):
        verified_results = json.dumps({
            "question": question,
            "orchestrator_plan": plan,
            "valid_results" : valid_results,
            })
        messages = [
            {"role": "system", "content": self.prompt},
            {"role": "user", "content": verified_results},
        ]
        response_schema = {
            "type": "json_schema",
            "json_schema": {
                "name": "synthesizer_result",
                "strict": True,
                "schema": {
                    "type": "object",
                    "properties": {
                        "confidence": {"type": "number"},
                        "answer": {"type": "string"},
                        "reasoning": {"type": "string"}
                    },
                    "required": ["confidence", "answer", "reasoning"],
                    "additionalProperties": False
                }
            }
        }

        params = {
            "model": ACTIVE_MODEL,
            "messages": messages,
            "response_format": response_schema,
            "temperature": 0,
        }
        response = client.chat.completions.create(**params)
        output = response.choices[0].message.content
        results = json.loads((output or "").strip())

        return AgentResult(
            agent_name="Synthesizer",
            answer=results["answer"],
            confidence=float(results["confidence"]),
            reasoning=results["reasoning"],
        )

In [143]:
SPECIALIST_PROMPTS = {
    "market_specialist": """
        You are market_specialist in a multi-agent stock analysis system.

        Your job is to answer only market- and price-related sub-tasks using the tools you have been given.

        Focus on:
        - price performance
        - momentum and trend
        - relative price comparisons
        - market open/closed status
        - top gainers, losers, and active stocks

        Avaliable tools:
        - get_tickers_by_sector:
        Use when the user asks about a sector/industry (e.g., energy, semiconductor, finance, tech stocks in DB). Returns all stocks in a sector or industry from the local database.
        - get_price_performance:
        Use for return comparisons over periods. Returns % price change for tickers over a period ('1mo','3mo','6mo','ytd','1y').
        - get_market_status:
        Use when asked whether markets/exchanges are open.
        - get_top_gainers_losers:
        Use for today’s top gainers/losers/most active.

        Rules:
        - Use only tool evidence, not guesses
        - Do not make claims about fundamentals, valuation, earnings, or news unless they are explicitly present in the provided tool output
        - If the task involves a sector or industry, first identify the relevant tickers before comparing price performance
        - If ranking is requested, ensure the ranking is based on the returned numbers
        - If data is missing or a tool fails, say so clearly
        - Keep the final answer concise and evidence-based
        - Include exact figures when available
        - Do not answer beyond the assigned task
        """,

    "fundamental_specialist": """
        You are fundamental_specialist in a multi-agent stock analysis system.

        Your job is to answer only fundamentals- and company-overview-related sub-tasks using the tools you have been given.

        Focus on:
        - P/E ratio
        - EPS
        - market cap
        - 52-week high and low
        - company-level comparisons
        - sector and exchange filtering through the local database

        Avaliable tools:
        - get_tickers_by_sector:
        Use when the user asks about a sector/industry (e.g., energy, semiconductor, finance, tech stocks in DB). Returns all stocks in a sector or industry from the local database.
        - query_local_db:
        Use for custom filtering/counting in stocks.db. Fields include ticker, company, sector, industry, market_cap (Large/Mid/Small), exchange. Make sure to use A valid SQL SELECT statement as input.
        - get_company_overview:
        Use for company fundamentals (P/E, EPS, market cap, 52-week high/low).


        Rules:
        - Use only tool evidence, not guesses
        - Do not make claims about recent price moves or news sentiment unless explicitly given in the task input
        - If the task involves selecting companies from a sector, exchange, or market-cap bucket, use the database tools first
        - When comparing companies, present the exact values clearly
        - If a metric is missing or a tool fails, say so explicitly
        - Keep the final answer concise and analytical
        - Do not answer beyond the assigned task
        """,

    "news_specialist": """
        You are news_specialist in a multi-agent stock analysis system.

        Your job is to answer only news- and sentiment-related sub-tasks using the tools you have been given.

        Focus on:
        - recent headlines
        - sentiment labels
        - sentiment scores
        - company-specific or sector-related news context

        Tools avaliable:
        - query_local_db:
        Use for custom filtering/counting in stocks.db. Fields include ticker, company, sector, industry, market_cap (Large/Mid/Small), exchange. Make sure to use A valid SQL SELECT statement as input.
        - get_news_sentiment:
        Use for sentiment/headlines for specific tickers.

        Rules:
        - Use only tool evidence, not guesses
        - Do not make claims about valuation, earnings, or price performance unless explicitly present in the provided data
        - If the task refers to a sector or group of companies, use the database tool only to identify relevant tickers
        - Summarize the most relevant headlines and sentiment clearly
        - If sentiment is mixed, say that explicitly
        - If there are too many possible tickers, prioritize the ones named in the task
        - If data is missing or a tool fails, say so clearly
        - Keep the answer concise and directly tied to the assigned task
        - Do not answer beyond the assigned task
        """,
}


In [162]:
def run_multi_agent(question, conv_hist = ""):
    t0 = time.perf_counter()

    MARKET_TOOLS      = [SCHEMA_TICKERS, SCHEMA_PRICE, SCHEMA_STATUS, SCHEMA_MOVERS]
    FUNDAMENTAL_TOOLS = [SCHEMA_OVERVIEW, SCHEMA_SQL, SCHEMA_TICKERS]
    SENTIMENT_TOOLS   = [SCHEMA_NEWS, SCHEMA_SQL]

    orchestrator = Orchestrator()
    market_specialist = Specialist("market_specialist", schema=MARKET_TOOLS)
    fundamental_specialist = Specialist("fundamental_specialist", schema=FUNDAMENTAL_TOOLS)
    news_specialist = Specialist("news_specialist", schema=SENTIMENT_TOOLS)
    critic = Critic()
    synthesizer = Synthsizer()

    SPECIALISTS = {
        "market_specialist": market_specialist,
        "fundamental_specialist": fundamental_specialist,
        "news_specialist": news_specialist,
    }
    SPECIALISTS_ANSWER = {}
    SPECIALISTS_RESULTS = []

    reply_is_ready = False
    max_attempts = 3

    while not reply_is_ready:
        orchestrator_results = orchestrator.run(question=question, conv_hist=conv_hist)
        plan = orchestrator_results.answer
        display(Markdown(plan))

        for specialist_called  in orchestrator_results.raw_data["specialist_to_call"]:
            sp = specialist_called["agent_name"]
            task = specialist_called["instruction"]
            display(Markdown(task))
            SPECIALISTS[sp].prompt = SPECIALIST_PROMPTS[sp]
            task_success = False
            task_attempt = 1
            cxt = ""

            while not task_success and task_attempt <= max_attempts:
                task_attempt += 1
                sp_results = SPECIALISTS[sp].run(task=task, cxt=cxt)
                critic_results = critic.run(task=task, specalist_results=sp_results)

                if not critic_results.answer:
                    cxt = f"\n\nYour last answer failed!\n\nlast answer:{sp_results.answer}\n\nfailure reason: {critic_results.reasoning}"
                    task_success = False
                else:
                    task_success = True
            
            if task_attempt > max_attempts:
                print(f"{sp} has reached it maximum attempts")
                SPECIALISTS_ANSWER[sp] = "Specialist failed in retrieving relevant infomation."
            else:
                print(f"{sp} has passed critic's evaluation")
                SPECIALISTS_ANSWER[sp] = sp_results.answer
            SPECIALISTS_RESULTS.append(sp_results)
    
        synthesizer_results = synthesizer.run(question=question, plan=plan, valid_results=SPECIALISTS_ANSWER)
        if synthesizer_results.confidence == 1.0:
            reply_is_ready = True
            print("reply is ready")
        else:
            reply_is_ready = False
            conv_hist += f"\n\nSpecalist results insufficient in answering user's question. Here's the reason:\n\n{synthesizer_results.reasoning}\n\nand replanning suggestions:\n\n{synthesizer_results.answer}"
            print("specialist results insufficent in answering user's question, returning to orchestrator replanning")

    wall_time = round(time.perf_counter() - t0, 3)
    return {
        "final_answer": synthesizer_results.answer,
        "agent_results": SPECIALISTS_RESULTS,
        "elapsed_sec": wall_time,
        "architecture": "orchestrator-critic"
    }




In [145]:
# Test 1 — check return contract
out1 = run_multi_agent("What is the P/E ratio of Apple (AAPL)?")
assert "final_answer"   in out1, "Missing final_answer key"
assert "agent_results"  in out1, "Missing agent_results key"
assert "elapsed_sec"    in out1, "Missing elapsed_sec key"
assert "architecture"   in out1, "Missing architecture key"
print(f"Architecture : {out1['architecture']}")
print(f"Agents ran   : {[r.agent_name for r in out1['agent_results']]}")
print(f"Answer       : {out1['final_answer'][:200]}")

To answer the user's question about the P/E ratio of Apple (AAPL), I will direct the request to the fundamental_specialist, as it is responsible for retrieving company overview data including the P/E ratio.

fundamental_specialist is calling tool: get_company_overview
Architecture : orchestrator-critic
Agents ran   : ['fundamental_specialist']
Answer       : The P/E ratio for Apple Inc. (AAPL) is 32.37. This metric indicates how much investors are willing to pay for each dollar of earnings, providing insight into the company's valuation relative to its ea


In [161]:
# Test 2 — cross-domain hard question
out2 = run_multi_agent("For the top 3 semiconductor stocks by 1-year return, what are their P/E ratios and current news sentiment?")
print(f"Architecture : {out2['architecture']}")
print(f"Agents ran   : {[r.agent_name for r in out2['agent_results']]}")
print(f"Tools used   : {[t for r in out2['agent_results'] for t in r.tools_called]}")
print(f"\nAnswer:\n{out2['final_answer'][:400]}")

To answer the user's question about the top 3 semiconductor stocks by 1-year return, I will first identify the top 3 semiconductor stocks based on their 1-year return using the market_specialist. Then, I will retrieve the P/E ratios for these stocks using the fundamental_specialist. Finally, I will gather the current news sentiment for these stocks using the news_specialist.

Identify the top 3 semiconductor stocks by 1-year return.

market_specialist is calling tool: get_tickers_by_sector
market_specialist is calling tool: get_price_performance
market_specialist has passed critic's evaluation


Retrieve the P/E ratios for the top 3 semiconductor stocks identified.

fundamental_specialist is calling tool: get_tickers_by_sector
fundamental_specialist is calling tool: get_company_overview
fundamental_specialist is calling tool: get_company_overview
fundamental_specialist is calling tool: get_company_overview
fundamental_specialist has passed critic's evaluation


...

To accurately answer the user's question regarding the top 3 semiconductor stocks by 1-year return, we will first identify the correct top 3 stocks. Then, we will retrieve the P/E ratios for these stocks from the fundamental_specialist and gather the current news sentiment for the same stocks from the news_specialist. This will ensure that the information provided is consistent and accurate.

Retrieve the P/E ratios for Micron Technology, Teradyne, and Applied Materials.

fundamental_specialist is calling tool: get_company_overview
fundamental_specialist is calling tool: get_company_overview
fundamental_specialist is calling tool: get_company_overview
fundamental_specialist has passed critic's evaluation


Retrieve the current news sentiment for Micron Technology, Teradyne, and Applied Materials.

news_specialist is calling tool: get_news_sentiment
news_specialist is calling tool: get_news_sentiment
news_specialist is calling tool: get_news_sentiment
news_specialist has passed critic's evaluation
reply is ready
Architecture : orchestrator-critic
Agents ran   : ['market_specialist', 'fundamental_specialist', 'news_specialist', 'market_specialist', 'fundamental_specialist', 'news_specialist', 'fundamental_specialist', 'news_specialist']
Tools used   : ['get_tickers_by_sector', 'get_price_performance', 'get_tickers_by_sector', 'get_company_overview', 'get_company_overview', 'get_company_overview', 'query_local_db', 'query_local_db', 'get_news_sentiment', 'get_news_sentiment', 'get_news_sentiment', 'get_tickers_by_sector', 'get_price_performance', 'get_company_overview', 'get_company_overview', 'get_company_overview', 'get_news_sentiment', 'get_news_sentiment', 'get_news_sentiment', 'get_company_overview', 'get_company_overview', 'get_company_overview', 'get_news_sentiment', 'get_ne

---
## 🛠️ Task 5 — Implement the LLM Evaluator (15 pts)

The evaluator is an **LLM-as-judge**: it reads a question, the expected answer description, and an agent's actual answer, then scores it.

This is how you measure accuracy across all architectures automatically — rather than reading 45 answers by hand.

---

### Required output format

Your evaluator must return a Python dict with exactly these keys.  
The evaluation runner reads all of them to fill the Excel sheet:

```python
{
    "score"                 : int,        # 0, 1, 2, or 3
    "max_score"             : int,        # always 3
    "reasoning"             : str,        # one sentence explaining the score
    "hallucination_detected": bool,       # True if the answer contains invented facts
    "key_issues"            : list[str],  # specific problems found; empty list if none
}
```

---

### Scoring rubric to include in your prompt

```
3 — Fully correct:    all required data present, numbers accurate, conditions met
2 — Partially correct: key data present but incomplete, gaps, or minor inaccuracies
1 — Mostly wrong:     attempted but wrong numbers, missed required conditions,
                      or claims that appear fabricated
0 — Complete failure: refused to answer, said data unavailable without trying tools,
                      or answer has no relevance to the question
```

---

### Hallucination detection — what to flag

Include these rules in your prompt:
- Specific numbers (prices, P/E ratios, % changes) with no tool data to support them
- Stock tickers that don't exist or aren't relevant to the question
- Definitive claims about "current" data without having called a live data tool

---

### Important considerations

**The evaluator only sees text** — it cannot verify numbers against live data.  
Its hallucination detection is based on reasoning about whether claims look fabricated,  
not on cross-checking against a ground truth source.  
You will reflect on this limitation in the graded questions.

**JSON parsing:** The LLM may wrap its response in markdown fences (```json ... ```).  
Strip those before parsing. Return the fallback dict on any parse error.

**Prompt placement:** Pass the rubric, the expected answer description, and the agent's  
actual answer all in one user message so the evaluator has full context.

---

### Calibration tests (provided below)

Three test cases check your evaluator before the full run:
1. A clearly correct answer (expect score = 3)
2. An answer with an invented number (expect `hallucination_detected = True`, score ≤ 1)
3. A refusal (expect score = 0)

All three must behave as expected before running the full evaluation.


In [ ]:
# ── YOUR EVALUATOR IMPLEMENTATION ────────────────────────────
#
# Things to decide:
#   - How detailed is your rubric? (more detail → more consistent scores)
#   - How do you instruct it to detect hallucinations vs honest uncertainty?
#   - How strict are you on partial answers? (affects SA vs MA comparison)
#   - How do you handle "I don't know" answers — 0 or 1?
#
# Required: function signature must be exactly as shown below.

def run_evaluator(question: str, expected_answer: str, agent_answer: str) -> dict:
    """
    Score one agent answer against the expected answer description.

    Returns dict with keys:
        score, max_score, reasoning, hallucination_detected, key_issues

    On JSON parse failure, return:
        {"score":0,"max_score":3,"reasoning":"evaluator parse error",
         "hallucination_detected":False,"key_issues":["evaluator failed to parse"]}
    """
    ### YOUR CODE HERE
    pass


# ── Calibration tests — all three must behave as expected ─────
print("=== Calibration Test 1 — correct answer (expect score=3) ===")
t1 = run_evaluator(
    question        = "What is the P/E ratio of Apple (AAPL)?",
    expected_answer = "Should return AAPL P/E ratio as a single numeric value from Alpha Vantage.",
    agent_answer    = "The current P/E ratio of Apple Inc. (AAPL) is 33.45.",
)
print(f"  Score: {t1['score']}/3 | Hallucination: {t1['hallucination_detected']}")
print(f"  Reasoning: {t1['reasoning']}")

print("\n=== Calibration Test 2 — fabricated number (expect hallucination=True, score≤1) ===")
t2 = run_evaluator(
    question        = "What is the P/E ratio of Apple (AAPL)?",
    expected_answer = "Should return AAPL P/E ratio as a single numeric value from Alpha Vantage.",
    agent_answer    = "Apple's P/E ratio is approximately 28.5 based on current market conditions.",
)
print(f"  Score: {t2['score']}/3 | Hallucination: {t2['hallucination_detected']}")
print(f"  Reasoning: {t2['reasoning']}")
assert t2["hallucination_detected"] == True, "Should detect fabricated P/E as hallucination"

print("\n=== Calibration Test 3 — refusal (expect score=0) ===")
t3 = run_evaluator(
    question        = "What is the P/E ratio of Apple (AAPL)?",
    expected_answer = "Should return AAPL P/E ratio as a single numeric value from Alpha Vantage.",
    agent_answer    = "I cannot retrieve real-time financial data. Please check Yahoo Finance.",
)
print(f"  Score: {t3['score']}/3 | Hallucination: {t3['hallucination_detected']}")
print(f"  Reasoning: {t3['reasoning']}")
assert t3["score"] == 0, "Refusal should score 0"

print("\n✅ Evaluator calibration complete")

## Step 5 — Benchmark Questions (Fixed — Do Not Modify)

15 questions across three difficulty levels. All three architectures run against all 15.

| Tier | Questions | What makes them harder |
|---|---|---|
| Easy (Q01–Q05) | 5 | 1 tool, single domain |
| Medium (Q06–Q10) | 5 | 2 tools, cross-domain reasoning |
| Hard (Q11–Q15) | 5 | 3+ tools, multi-condition filtering or cross-domain synthesis |


In [20]:
BENCHMARK_QUESTIONS = [
    # ── EASY ──────────────────────────────────────────────────────────────
    {"id":"Q01","complexity":"easy","category":"sector_lookup",
     "question":"List all semiconductor companies in the database.",
     "expected":"Should return company names and tickers for semiconductor stocks from the local DB. "
                "Tickers include NVDA, AMD, INTC, QCOM, AVGO, TXN, ADI, MU and others."},
    {"id":"Q02","complexity":"easy","category":"market_status",
     "question":"Are the US stock markets open right now?",
     "expected":"Should return the current open/closed status for NYSE and NASDAQ "
                "with their trading hours."},
    {"id":"Q03","complexity":"easy","category":"fundamentals",
     "question":"What is the P/E ratio of Apple (AAPL)?",
     "expected":"Should return AAPL P/E ratio as a single numeric value fetched from Alpha Vantage."},
    {"id":"Q04","complexity":"easy","category":"sentiment",
     "question":"What is the latest news sentiment for Microsoft (MSFT)?",
     "expected":"Should return 3–5 recent MSFT headlines with Bullish/Bearish/Neutral labels and scores."},
    {"id":"Q05","complexity":"easy","category":"price",
     "question":"What is NVIDIA's stock price performance over the last month?",
     "expected":"Should return NVDA start price, end price, and % change for the 1-month period."},

    # ── MEDIUM ─────────────────────────────────────────────────────────────
    {"id":"Q06","complexity":"medium","category":"price_comparison",
     "question":"Compare the 1-year price performance of AAPL, MSFT, and GOOGL. Which grew the most?",
     "expected":"Should fetch 1y performance for all 3 tickers, return % change for each, "
                "and identify the highest performer."},
    {"id":"Q07","complexity":"medium","category":"fundamentals",
     "question":"Compare the P/E ratios of AAPL, MSFT, and NVDA. Which looks most expensive?",
     "expected":"Should return P/E ratios for all 3 tickers and identify which has the highest P/E."},
    {"id":"Q08","complexity":"medium","category":"sector_price",
     "question":"Which energy stocks in the database had the best 6-month performance?",
     "expected":"Should query the DB for energy sector tickers, fetch 6-month price performance "
                "for each, and return them ranked by % change."},
    {"id":"Q09","complexity":"medium","category":"sentiment",
     "question":"What is the news sentiment for Tesla (TSLA) and how has its stock moved this month?",
     "expected":"Should return TSLA news sentiment (label + score) AND 1-month price % change "
                "from two separate tool calls."},
    {"id":"Q10","complexity":"medium","category":"fundamentals",
     "question":"What are the 52-week high and low for JPMorgan (JPM) and Goldman Sachs (GS)?",
     "expected":"Should return 52-week high and low for both JPM and GS fetched from Alpha Vantage."},

    # ── HARD ───────────────────────────────────────────────────────────────
    {"id":"Q11","complexity":"hard","category":"multi_condition",
     "question":"Which tech stocks dropped this month but grew this year? Return the top 3.",
     "expected":"Should get tech tickers from DB, fetch both 1-month and year-to-date performance, "
                "filter for negative 1-month AND positive YTD, return top 3 by yearly growth with "
                "exact percentages. Results must satisfy both conditions simultaneously."},
    {"id":"Q12","complexity":"hard","category":"multi_condition",
     "question":"Which large-cap technology stocks on NASDAQ have grown more than 20% this year?",
     "expected":"Should query DB for large-cap NASDAQ tech stocks, fetch YTD performance, "
                "filter for >20% growth, and return matching tickers with exact % change."},
    {"id":"Q13","complexity":"hard","category":"cross_domain",
     "question":"For the top 3 semiconductor stocks by 1-year return, what are their P/E ratios "
                "and current news sentiment?",
     "expected":"Should find semiconductor tickers in DB, rank by 1-year return to find top 3, "
                "then fetch P/E ratio AND news sentiment for each — requiring three separate "
                "data domains (price, fundamentals, sentiment)."},
    {"id":"Q14","complexity":"hard","category":"cross_domain",
     "question":"Compare the market cap, P/E ratio, and 1-year stock performance of JPM, GS, and BAC.",
     "expected":"Should return market cap, P/E, and 1-year % change for all 3 tickers, "
                "combining Alpha Vantage fundamentals and yfinance price data."},
    {"id":"Q15","complexity":"hard","category":"multi_condition",
     "question":"Which finance sector stocks are trading closer to their 52-week low than their "
                "52-week high? Return the news sentiment for each.",
     "expected":"Should get finance sector tickers from DB, fetch 52-week high and low for each, "
                "compute proximity to the low, then fetch news sentiment for qualifying stocks."},
]

print(f"✅ {len(BENCHMARK_QUESTIONS)} questions loaded")
for tier in ["easy","medium","hard"]:
    count = sum(1 for q in BENCHMARK_QUESTIONS if q["complexity"]==tier)
    print(f"   {tier:8s}: {count} questions")

✅ 15 questions loaded
   easy    : 5 questions
   medium  : 5 questions
   hard    : 5 questions


## Step 6 — Evaluation Runner and Excel Writer (Provided)

This runner calls all three architectures on all 15 questions, evaluates each answer,  
and writes results to an Excel file with two sheets:

- **Results** — one row per question with all metrics for all three architectures  
- **Summary** — accuracy % by architecture and difficulty tier (auto-calculated)

Results are saved after every question — if the run crashes, you do not lose progress.

### Excel columns produced

| Column group | Columns written |
|---|---|
| Question | ID, complexity, category, question text, expected answer |
| Baseline | answer, time(s), eval score (0-3), eval reasoning, hallucination detected, issues |
| Single Agent | answer, tools used, tool count, iterations, time(s), eval score, reasoning, hallucination, issues |
| Multi Agent | answer, tools used, tool count, time(s), confidence, critic issues, agents activated, architecture, eval score, reasoning, hallucination, issues |


In [21]:
@dataclass
class EvalRecord:
    # Question
    question_id : str;  question    : str;  complexity : str
    category    : str;  expected    : str
    # Baseline
    bl_answer       : str   = "";  bl_time         : float = 0.0
    bl_score        : int   = -1;  bl_reasoning    : str   = ""
    bl_hallucination: str   = "";  bl_issues       : str   = ""
    # Single agent
    sa_answer       : str   = "";  sa_tools        : str   = ""
    sa_tool_count   : int   = 0;   sa_iters        : int   = 0
    sa_time         : float = 0.0; sa_score        : int   = -1
    sa_reasoning    : str   = "";  sa_hallucination: str   = ""
    sa_issues       : str   = ""
    # Multi agent
    ma_answer       : str   = "";  ma_tools        : str   = ""
    ma_tool_count   : int   = 0;   ma_time         : float = 0.0
    ma_confidence   : str   = "";  ma_critic_issues: int   = 0
    ma_agents       : str   = "";  ma_architecture : str   = ""
    ma_score        : int   = -1;  ma_reasoning    : str   = ""
    ma_hallucination: str   = "";  ma_issues       : str   = ""


# ── Column rename map (internal name → Excel header) ─────────
_COL_NAMES = {
    "question_id":"Question ID","question":"Question","complexity":"Difficulty",
    "category":"Category","expected":"Expected Answer",
    "bl_answer":"Baseline Answer","bl_time":"Baseline Time (s)",
    "bl_score":"Baseline Score /3","bl_reasoning":"Baseline Eval Reasoning",
    "bl_hallucination":"Baseline Hallucination","bl_issues":"Baseline Issues",
    "sa_answer":"SA Answer","sa_tools":"SA Tools Used","sa_tool_count":"SA Tool Count",
    "sa_iters":"SA Iterations","sa_time":"SA Time (s)",
    "sa_score":"SA Score /3","sa_reasoning":"SA Eval Reasoning",
    "sa_hallucination":"SA Hallucination","sa_issues":"SA Issues",
    "ma_answer":"MA Answer","ma_tools":"MA Tools Used","ma_tool_count":"MA Tool Count",
    "ma_time":"MA Time (s)","ma_confidence":"MA Avg Confidence",
    "ma_critic_issues":"MA Critic Issue Count","ma_agents":"MA Agents Activated",
    "ma_architecture":"MA Architecture",
    "ma_score":"MA Score /3","ma_reasoning":"MA Eval Reasoning",
    "ma_hallucination":"MA Hallucination","ma_issues":"MA Issues",
}


def _save_excel(records: list, path: str):
    df = pd.DataFrame([r.__dict__ for r in records]).rename(columns=_COL_NAMES)

    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        # ── Sheet 1: full results ──────────────────────────────
        df.to_excel(writer, index=False, sheet_name="Results")

        # ── Sheet 2: summary by architecture × difficulty ──────
        rows = []
        for arch, sc, tc, hc in [
            ("Baseline",     "Baseline Score /3", "Baseline Time (s)", "Baseline Hallucination"),
            ("Single Agent", "SA Score /3",       "SA Time (s)",       "SA Hallucination"),
            ("Multi Agent",  "MA Score /3",       "MA Time (s)",       "MA Hallucination"),
        ]:
            for tier in ["easy", "medium", "hard", "all"]:
                subset = df if tier == "all" else df[df["Difficulty"] == tier]
                valid  = subset[subset[sc] >= 0]
                avg_s  = valid[sc].mean() if len(valid) else 0
                rows.append({
                    "Architecture"   : arch,
                    "Difficulty"     : tier,
                    "Questions Scored": len(valid),
                    "Avg Score /3"   : round(avg_s, 2),
                    "Accuracy %"     : round(avg_s / 3 * 100, 1),
                    "Avg Time (s)"   : round(df[tc].mean(), 1),
                    "Hallucinations" : (df[hc] == "True").sum(),
                })
        pd.DataFrame(rows).to_excel(writer, index=False, sheet_name="Summary")


def run_full_evaluation(output_xlsx: str = "results.xlsx", delay_sec: float = 3.0):
    """
    Run all 15 questions through baseline, single agent, and multi agent.
    Score each answer. Write results to Excel after every question.
    """
    records = []
    total   = len(BENCHMARK_QUESTIONS)
    print(f"\n{'='*62}")
    print(f"  FULL EVALUATION  |  {total} questions × 3 architectures")
    print(f"  Model: {ACTIVE_MODEL}  |  Output: {output_xlsx}")
    print(f"{'='*62}\n")

    for i, q in enumerate(BENCHMARK_QUESTIONS, 1):
        print(f"[{i:02d}/{total}] {q['id']} ({q['complexity']:6s}) {q['question'][:52]}...")
        rec = EvalRecord(question_id=q["id"], question=q["question"],
                         complexity=q["complexity"], category=q["category"],
                         expected=q["expected"])

        # ── Baseline ───────────────────────────────────────────
        print("         baseline  ...", end=" ", flush=True)
        try:
            t0 = time.time()
            bl = run_baseline(q["question"], verbose=False)
            rec.bl_answer = bl.answer.replace("\n", " ")
            rec.bl_time   = round(time.time() - t0, 2)
            ev = run_evaluator(q["question"], q["expected"], bl.answer)
            rec.bl_score        = ev.get("score", -1)
            rec.bl_reasoning    = ev.get("reasoning", "")
            rec.bl_hallucination= str(ev.get("hallucination_detected", False))
            rec.bl_issues       = " | ".join(ev.get("key_issues", []))
            print(f"✅  {rec.bl_time:5.1f}s  score {rec.bl_score}/3")
        except Exception as e:
            print(f"❌  {e}")

        # ── Single Agent ───────────────────────────────────────
        print("         single    ...", end=" ", flush=True)
        try:
            t0 = time.time()
            sa = run_single_agent(q["question"], verbose=False)
            rec.sa_answer    = sa.answer.replace("\n", " ")
            rec.sa_tools     = ", ".join(sa.tools_called)
            rec.sa_tool_count= len(sa.tools_called)
            rec.sa_iters     = len(sa.tools_called) + 1   # approx
            rec.sa_time      = round(time.time() - t0, 2)
            ev = run_evaluator(q["question"], q["expected"], sa.answer)
            rec.sa_score        = ev.get("score", -1)
            rec.sa_reasoning    = ev.get("reasoning", "")
            rec.sa_hallucination= str(ev.get("hallucination_detected", False))
            rec.sa_issues       = " | ".join(ev.get("key_issues", []))
            print(f"✅  {rec.sa_time:5.1f}s  score {rec.sa_score}/3"
                  f"  tools [{rec.sa_tools or 'none'}]")
        except Exception as e:
            print(f"❌  {e}")

        # ── Multi Agent ────────────────────────────────────────
        print("         multi     ...", end=" ", flush=True)
        try:
            t0  = time.time()
            ma  = run_multi_agent(q["question"], verbose=False)
            res = ma.get("agent_results", [])
            all_tools  = [t for r in res for t in r.tools_called]
            all_issues = [iss for r in res for iss in r.issues_found]
            avg_conf   = sum(r.confidence for r in res) / len(res) if res else 0.0
            rec.ma_answer        = ma["final_answer"].replace("\n", " ")
            rec.ma_tools         = ", ".join(dict.fromkeys(all_tools))
            rec.ma_tool_count    = len(all_tools)
            rec.ma_time          = round(time.time() - t0, 2)
            rec.ma_confidence    = f"{avg_conf:.0%}"
            rec.ma_critic_issues = len(all_issues)
            rec.ma_agents        = ", ".join(r.agent_name for r in res)
            rec.ma_architecture  = ma.get("architecture", "")
            ev = run_evaluator(q["question"], q["expected"], ma["final_answer"])
            rec.ma_score        = ev.get("score", -1)
            rec.ma_reasoning    = ev.get("reasoning", "")
            rec.ma_hallucination= str(ev.get("hallucination_detected", False))
            rec.ma_issues       = " | ".join(ev.get("key_issues", []))
            print(f"✅  {rec.ma_time:5.1f}s  score {rec.ma_score}/3"
                  f"  conf {rec.ma_confidence}  issues {rec.ma_critic_issues}")
        except Exception as e:
            print(f"❌  {e}")

        records.append(rec)
        _save_excel(records, output_xlsx)       # save progress after every question

        if i < total:
            print(f"         ⏳ waiting {delay_sec}s ...\n")
            time.sleep(delay_sec)

    # ── Print summary table ────────────────────────────────────
    print(f"\n{'='*62}  RESULTS")
    print(f"{'Architecture':<18} {'Easy':>8} {'Medium':>8} {'Hard':>8} {'Overall':>8}")
    print("─" * 52)
    for arch, sk in [("Baseline","bl_score"),("Single Agent","sa_score"),("Multi Agent","ma_score")]:
        def pct(tier):
            s = [getattr(r, sk) for r in records
                 if getattr(r, sk) >= 0 and (tier=="all" or r.complexity==tier)]
            return f"{sum(s)/len(s)/3*100:.0f}%" if s else "—"
        print(f"{arch:<18} {pct('easy'):>8} {pct('medium'):>8} {pct('hard'):>8} {pct('all'):>8}")

    print(f"\n✅ Saved → {output_xlsx}")
    return output_xlsx

print("✅ Evaluation runner ready")

✅ Evaluation runner ready


## Step 7 — Run the Evaluation

Run the sanity check cell first. Only proceed to the full evaluation once all three architectures  
produce valid output on the test question.


In [1]:
# ── Sanity check — one question, all three architectures ──────
q_test = BENCHMARK_QUESTIONS[2]        # Q03 — easy fundamentals
print(f"Test question: {q_test['question']}\n")

bl_t = run_baseline(q_test["question"], verbose=False)
sa_t = run_single_agent(q_test["question"], verbose=False)
ma_t = run_multi_agent(q_test["question"], verbose=False)

print(f"Baseline     : {bl_t.answer[:120]}")
print(f"Single Agent : {sa_t.answer[:120]}  |  tools: {sa_t.tools_called}")
print(f"Multi Agent  : {ma_t['final_answer'][:120]}  |  arch: {ma_t['architecture']}")

ev_bl = run_evaluator(q_test["question"], q_test["expected"], bl_t.answer)
ev_sa = run_evaluator(q_test["question"], q_test["expected"], sa_t.answer)
ev_ma = run_evaluator(q_test["question"], q_test["expected"], ma_t["final_answer"])
print(f"\nScores — Baseline: {ev_bl['score']}/3  |  Single: {ev_sa['score']}/3  |  Multi: {ev_ma['score']}/3")

NameError: name 'BENCHMARK_QUESTIONS' is not defined

In [ ]:
# ── Full evaluation — gpt-4o-mini ─────────────────────────────
ACTIVE_MODEL = MODEL_SMALL
run_full_evaluation(output_xlsx="results_gpt4o_mini.xlsx", delay_sec=3.0)

In [ ]:
# ── Full evaluation — gpt-4o (required for reflection Q4) ─────
ACTIVE_MODEL = MODEL_LARGE
run_full_evaluation(output_xlsx="results_gpt4o.xlsx", delay_sec=3.0)

---
## 📝 Graded Reflection Questions (30 pts)

Answer each question in the markdown cell below it.  
Every answer must reference specific question IDs and scores from your Excel results.  
Vague answers without data will receive at most half marks.


### Q0 — Compare the performance of baseline vs single agent implementation? (5 pts)
- Analyse whether the usecase needs the agentic implementation, if yes why? if no, why not?
**Your answer (minimum 150 words, cite question IDs and scores):**

>The single agent performs better than the baseline for this finance use case because it can use tools to fetch current, grounded data. The clearest example is the Apple P/E question: the baseline gives only an approximate answer with no evidence, while r1 returns a specific value using get_company_overview. The same trend appears in r4, where the single agent gives NVIDIA’s actual 1-month price change with start and end prices. It is less perfect on more complex tasks: r2 is useful but not well ranked, and r3 struggles on multi-condition filtering. Overall, this use case does benefit from an agentic implementation because finance questions often require live numbers, database lookup, or chained tool use that the baseline cannot do reliably.

### Q1 — Is Multi-Agent Actually Necessary? (5 pts)

Using your `results_gpt4o_mini.xlsx`:

- For which difficulty tier (easy / medium / hard) does multi-agent most outperform single agent?
- For which tier is the difference smallest?
- Give **2 concrete examples** — one where multi-agent clearly won, one where single agent was equivalent or better.  
  For each example, cite the question ID, both scores, and explain *why the architecture made the difference*.

**Your answer (minimum 150 words, cite question IDs and scores):**

>  Absolutely. As what I can tell from the 5 question in Task 3, A single agent can handle easy questions well, such as r1 and r4, and can also manage some medium questions like r5. The main weakness appears on more complex reasoning tasks. In r2, the agent retrieves the right type of data but does not rank it cleanly. While in r3, the hard question, it fail to answer the questions clearly. It list a few stock that is not part of answer and then give the conclusion. Note this is already improve after I put a dedicated prompt for it to check whether its answer matches with what the question asks. Otherwise, it can go completely wrong as there is probably no such stock met the question criteria in the database. 


### Q2 — Is the Evaluator Reliable? (5 pts)

Find **3 questions** in your results where you disagree with the score the evaluator assigned.  
For each one:
- What score did it give, and what score do you think is correct?
- Why did it get it wrong — did it miss a hallucination, or penalise an answer that was actually correct?

Then answer: is your evaluator systematically biased in any direction?  
What would you change in your prompt to fix the biggest problem you found?

**Your answer (minimum 150 words):**

> *Replace this text.*


### Q3 — Accuracy Across Architectures and Difficulty Tiers (5 pts)

Fill in the table below from your `results_gpt4o_mini.xlsx` Summary sheet, then write your analysis.

| Architecture | Easy % | Medium % | Hard % | Overall % |
|---|---|---|---|---|
| Baseline | | | | |
| Single Agent | | | | |
| Multi Agent | | | | |

- Where does each architecture most significantly break down?
- Is the accuracy drop from medium → hard larger for some architectures than others?
- What does this tell you about *which type of question* benefits most from an agentic approach?

**Your analysis (minimum 100 words):**

> *Replace this text.*


### Q4 — gpt-4o-mini vs gpt-4o with Your Multi-Agent (5 pts)

Compare `results_gpt4o_mini.xlsx` and `results_gpt4o.xlsx` for the **multi-agent architecture only**.

- Which question categories show the biggest accuracy improvement with the larger model?
- Does the confidence score (or critic issue count) change meaningfully?
- On hard questions specifically — is the accuracy difference large enough to justify the cost?
- Is there any category where the smaller model is actually sufficient?

**Your answer (minimum 150 words):**

> *Replace this text.*


### Q5 — Your Multi-Agent Design Decisions (5 pts)

Document the architecture you built:

1. Which pattern did you choose (orchestrator-critic, pipeline, parallel, or hybrid)? Why?
2. How did you divide tools between specialists? Explain each agent's scope.
3. What is your verification / confidence mechanism and how does it work?
4. What did you try first that did not work, and what did you change?
5. Looking at your results — did your architecture actually reduce hallucinations compared to the single agent? Show the numbers.

**Your answer (minimum 200 words):**

> *Replace this text.*


---
## ✅ Submission Checklist

- [ ] `get_company_overview()` — all assertions in Task 1 pass
- [ ] `get_tickers_by_sector()` — sector match AND industry fallback working
- [ ] `run_baseline()` — `tools_called == []`, answer not empty
- [ ] `run_single_agent()` — uses tools, system prompt reasoning documented in comments
- [ ] `run_multi_agent()` — returns correct dict contract, architecture documented in comments
- [ ] `run_evaluator()` — all 3 calibration tests pass
- [ ] `results_gpt4o_mini.xlsx` — Results + Summary sheets filled for all 15 questions
- [ ] `results_gpt4o.xlsx` — Results + Summary sheets filled for all 15 questions
- [ ] All 5 reflection questions answered with question IDs and scores cited

**Submit:** this notebook + `results_gpt4o_mini.xlsx` + `results_gpt4o.xlsx` + insights.doc/pdf (explaning design choices)
